# Access from DeepMarkets
Only use this for real run. Otherwise, use limit_order_simulation.py

The parquet is saved to the current working directory when the ABIDES config runs:
`simulator/DeepMarket/ABIDES/limit_order_abides_results.parquet`

Run the simulation first:
```bash
cd simulator/DeepMarket/ABIDES
python config/crypto_limit_order_sim.py
```

In [1]:
import pandas as pd
import glob
import os

# The parquet is saved to the CWD when the ABIDES config runs.
# From this notebook (research/), the expected path is:
expected_path = '../simulator/DeepMarket/ABIDES/limit_order_abides_results.parquet'

if os.path.exists(expected_path):
    path = expected_path
else:
    # Fallback: search the whole project for the file
    matches = glob.glob('**/limit_order_abides_results.parquet', recursive=True)
    if matches:
        path = matches[0]
        print(f"Found parquet at: {path}")
    else:
        raise FileNotFoundError(
            "limit_order_abides_results.parquet not found. "
            "Run the ABIDES simulation first: "
            "cd simulator/DeepMarket/ABIDES && python config/crypto_limit_order_sim.py"
        )

df = pd.read_parquet(path)
print(f"Loaded {len(df)} rows from {path}")
print(f"Columns: {list(df.columns)}")
print()

summary = df.groupby('qty').agg(
    total=('both_filled', 'count'),
    filled=('both_filled', 'sum'),
    fill_rate=('both_filled', 'mean'),
    avg_net_spread=('net_spread', 'mean'),
    avg_ttf=('time_to_fill_sec', 'mean'),
)
print(summary)

Loaded 954 rows from ../simulator/DeepMarket/ABIDES/limit_order_abides_results.parquet
Columns: ['signal_time', 'entry_z', 'direction', 'qty', 'method', 'perp_limit_price', 'spot_limit_price', 'perp_filled', 'spot_filled', 'both_filled', 'perp_fill_price', 'spot_fill_price', 'perp_fill_time', 'spot_fill_time', 'perp_fill_qty', 'spot_fill_qty', 'time_to_fill_sec', 'cancelled', 'captured_spread', 'fee_cost', 'fee_bps', 'net_spread']

     total  filled  fill_rate  avg_net_spread  avg_ttf
qty                                                   
1.0    954       1   0.001048         21.7995     3.38


## Notes on the results

- `both_filled`: True if BOTH legs (perp + spot) were fully filled within the 5s timeout
- `fill_rate`: fraction of signals where both legs filled — this is your scalability metric
- `avg_net_spread`: per-unit net spread (captured spread minus per-unit fees) — should be positive for profitability
- `avg_ttf`: average time-to-fill in seconds

**The scalability ceiling** is the largest `qty` where `fill_rate` is still acceptable (e.g. >50%) AND `avg_net_spread > 0`.

**Important**: If you ran with `--qty X` (single qty per simulation), each run overwrites the parquet. Run each qty separately and combine the results, or run the full grid (no `--qty` flag) for all sizes at once.